In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import pandas as pd
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from pathlib import Path
sys.path.append(os.path.abspath(".."))


from dotenv import load_dotenv
# Load environment variables
load_dotenv()

from src.portfolio_analytics import run_portfolio_analytics
from src.real_estate_bonds_data_builder import PortfolioMarketSimulator
from src.asset_class_builder import consolidate_portfolio_data
from src.market_data_engine import update_and_fetch_market_data
from src.index_builder import build_equal_weight_indices

# --- SENSITIVE LOGIC COMMENTED OUT ---
from src.etf1_synthetic_index_creator import (build_synthetic_index_engine,create_etf_file,inspect_csv_format,plot_synthetic_vs_live_validation,calculate_model_probability)

In [ ]:
#base_dir = Path(os.getenv("data_dir", "."))
#processed_dir = os.path.join(base_dir, "processed")
market_data_dir = Path("../data/raw/market_data_for_risk_analysis")
config_dir = Path("../config")
chart_dir = Path("../data/charts")
processed_dir = Path("../data/processed")
risk_analysis_dir = Path("../data/processed/risk_analysis")

yfinance_ticker_mapper = Path(config_dir) / "yfinance_ticker_mapper.csv"
stocks_data = Path(risk_analysis_dir) / "market_features_master.csv"
etf1 = Path(processed_dir) / "ETF1_holdings_enriched.csv"
Consolidated_Portfolio_Positions = Path(processed_dir) / "Consolidated_Portfolio_Positions.csv"

## 1. Synthetic Data Generation (Real Estate & Fixed Income)

### Context & Structural Limitations
Before proceeding to portfolio optimization, we must address critical data gaps in two asset classes: **Physical Real Estate** and **Bonds**. 

1. **Real Estate Opacity:** High-frequency, mark-to-market transaction data for localized physical real estate (e.g., tier-2/tier-3 regions or specific land segments) is virtually non-existent or highly illiquid. Publicly traded equity proxies (like REITs or developer stocks) are heavily bound to stock market sentiment and earnings volatility, failing to reflect the smooth, low-correlation profile of physical land.
2. **Fixed Income/Bonds:** Retail bond data often lacks continuous, daily exchange-traded pricing histories matching the timeline of liquid equities.

### Modeling Approach: Geometric Brownian Motion (GBM)
To ensure the data processing pipeline runs seamlessly without shape or frequency mismatches, we use a **Geometric Brownian Motion (GBM)** framework to synthesize a daily return time-series for these two assets, following the continuous-time stochastic path generation outlined by **Reddy & Clinton (2016)**. 

$$\Delta S = S \cdot (\mu \cdot \Delta t + \sigma \cdot \epsilon \cdot \sqrt{\Delta t})$$

* **Asset Parameters:** The generation relies on estimated macro metrics (annual appreciation rate $\mu$ and annualized volatility $\sigma$) provided in `portfolio_platform5_input.csv`.
* **Role in the Model:** Given the inherent limitations of tracking "dark" or illiquid asset transactions, these generated paths are mathematically treated as localized noise vectors. They provide a statistically sound baseline for variance and cross-asset covariance matrices while acting as a stable, low-correlation risk anchor within the broader multi-asset framework.

> **Academic Citation:** Reddy, K., & Clinton, V. (2016). Simulating Stock Prices Using Geometric Brownian Motion: Evidence from Australian Companies. *Australasian Accounting, Business and Finance Journal*, 10(3), 23-47.

*The following code blocks synthesize these high-frequency daily datasets over the designated 5-year lookback period.*

In [ ]:
INPUT_CSV = Path(processed_dir) / "portfolio_platform5_input.csv"

# Initialize our dynamic tracking engine
simulator = PortfolioMarketSimulator(
    input_csv_path=INPUT_CSV,
    processed_dir=processed_dir,
    market_data_dir=market_data_dir,
    start_date='2021-06-20',
    end_date='2026-06-26'
)

# Run pipeline execution across target assets
simulator.process_portfolio()

# Target directory verification check
asset_class_dir = os.path.join(processed_dir, "Asset class")
target_prefixes = ('real_estate', 'bonds')

# Ensure the directory exists before scanning
if os.path.exists(asset_class_dir):
    generated_files = [
        f for f in os.listdir(asset_class_dir) 
        if f.lower().startswith(target_prefixes) and f.endswith('.csv')
    ]
else:
    generated_files = []


# 3. Clean Web-App / Terminal Notification
print("\n" + "="*75)
if len(generated_files) > 0:
    print(f"SUCCESS: New synthetic data streams for Real Estate and Bonds generated successfully!")
    print(f"Files saved to: {asset_class_dir}")
    print(f"Generated datasets: {', '.join(generated_files)}")
else:
    print("WARNING: Pipeline executed, but no Real Estate or Bonds files were detected.")
print("="*75)

### Data Pipeline: Central FX and Asset Synchronization
This section manages the two-phase update process for your market database:
1. **Central FX Synchronization:** Checks for the latest `EURUSD` exchange rates, fetches missing daily data via `yfinance`, and maintains a master CSV for consistent currency translation.
2. **Asset Update Pipeline:** Iterates through your ticker mapper list to either fetch new 10-year datasets or incrementally update existing asset CSVs. It automatically detects asset currency (USD vs. EUR), applies real-time conversion factors using the synchronized FX series, and ensures data integrity by dropping duplicates and standardizing date formats before saving.

In [ ]:
update_and_fetch_market_data()

### Rationale for Synthetic Index Construction

**Research Objective:** The target Source 1 Etf's Index (please check .env file to know the etf name) is a newly established financial product, with formal live performance data available for only the past ~15 months. For rigorous academic and financial research, a 15-month timeframe is statistically insufficient to evaluate the sector's performance, volatility, and behavior across different macroeconomic cycles.

**Methodology:**
To overcome this data limitation, we constructed a **Synthetic Proxy Index**. By extracting the current constituent weights of the ETF and back-testing their historical market data over a 10-year horizon, we establish a robust, mathematical proxy. This allows us to:
1. Conduct long-term trend and risk analysis (5–10 years) beyond the limitations of the live ETF inception date.
2. Validate the sector's historical resilience prior to recent geopolitical catalysts.
3. Perform out-of-sample correlation testing between our synthetic model and the live ETF during their 15-month overlap to prove the model's high-fidelity tracking accuracy.

In [ ]:
# --- SENSITIVE LOGIC COMMENTED OUT ---
# Input File references
etf_file = processed_dir / "ETF1_holdings_enriched_tickers_adjusted.csv"
live_etf_file = os.getenv("Etf1_original_data")
output_directory = processed_dir / "index_engine_outputs"

## 1. Run Core Calculations Backtest Engine
index_results = build_synthetic_index_engine(
    etf_holdings_path=etf_file,
    live_etf_path=live_etf_file,
    output_dir=output_directory
)

#print("\n" + "="*50 + "\n")

## 2. Reformat Data to Target Structure File Setup
create_etf_file(
   index_engine_dir=output_directory,
    market_data_dir=market_data_dir
)

#print("\n" + "="*50 + "\n")

## 3. Post-Export Sanity Diagnostics Check
inspect_csv_format(os.getenv("Etf1_data"))

#print("\n" + "="*50 + "\n")

## 4. Generate Performance Comparison Graphs
plot_synthetic_vs_live_validation(index_results, live_etf_file)

#print("\n" + "="*50 + "\n")

## 5. Measure Statistical Variance Probability 
calculate_model_probability(
    index_results, 
    live_etf_file, 
    margin_of_error=0.001
)

### Data Consolidation Pipeline
This step automates the preparation of raw market data. It maps holdings to ticker identifiers and asset classes, aligns historical price timeframes to ensure a common index, and generates organized `_Consolidated.csv` files for each asset class in the `/Asset class` directory.

In [ ]:
output_dir = Path(processed_dir) / "Asset class"

consolidate_portfolio_data(
    positions_path=Consolidated_Portfolio_Positions,
    mapper_path=yfinance_ticker_mapper,
    market_data_dir=market_data_dir,
    output_dir=output_dir,
    price_column='Adjusted close price'
)

### Equal-Weight Index Construction
This function processes consolidated asset data to generate performance benchmarks. It calculates daily returns for each asset, computes an equal-weighted average for each asset class, and normalizes the resulting performance series to a base value of 100. The outputs are saved as `_EW_Index.csv` files to be used for macro-level analysis.

In [ ]:
build_equal_weight_indices(output_dir)

### Portfolio Risk Analytics Dashboard Creator

This module generates an interactive, multi-asset risk analytics dashboard. It processes raw financial return series, computes performance metrics, and builds an exportable HTML risk report.

#### Core Functionality:
* **Dynamic Calculations:** Computes annualized returns, volatilities, and Sharpe ratios dynamically based on asset-specific trading day frequencies (e.g., 252 days for equities, 365 days for crypto/bonds).
* **Cross-Asset Correlation:** Builds asset-level and macro correlation matrices with auto-adjusting margins and adaptive sizing to ensure legibility.
* **Risk Visualization:** Generates interactive scatter plots (Risk vs. Return) and histograms (Daily Return Distributions) using Plotly.
* **Automated Reporting:** Compiles charts alongside responsive data tables into an interactive HTML dashboard (`Master_Risk_Dashboard.html`).

In [ ]:
run_portfolio_analytics(processed_dir, chart_dir,Consolidated_Portfolio_Positions, yfinance_ticker_mapper)